# Limpieza y preprocesamiento de datos

Este notebook contiene los pasos de limpieza aplicados a los archivos raw del proyecto. La limpieza se realizara de forma progresiva, documentando cada transformacion antes de guardar resultados finales. El primer paso consiste en estandarizar los nombres de pacientes para facilitar comparaciones, deduplicacion y fusion posterior.

## 1. Configuracion inicial

Se cargan las librerias necesarias y se definen las rutas de entrada y salida. Los datos originales se leen desde `Datos Raw`; las versiones limpias se guardaran posteriormente en una carpeta separada para no modificar las fuentes originales.

In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)

BASE_DIR = Path.cwd().parent if Path.cwd().name == '04_Limpieza_y_Preprocesamiento' else Path.cwd()
RAW_DIR = BASE_DIR / '02_Datos_Crudos'
OUTPUT_DIR = BASE_DIR / '04_Limpieza_y_Preprocesamiento'

print('Carpeta raw:', RAW_DIR)
print('Carpeta de trabajo:', OUTPUT_DIR)

## 2. Carga consolidada de archivos

Se cargan todos los archivos de cada sistema en dataframes consolidados. Adicionalmente, se agregan columnas de trazabilidad (`archivo`, `sede`, `anio`) para conservar el origen de cada registro durante la limpieza.

In [ ]:
FILE_RE = re.compile(r'^(transacciones|clinica|prescripciones)_(mexico|estados_unidos|espana)_(2023|2024|2025)\.csv$')

def cargar_sistema(sistema):
    frames = []
    for path in sorted(RAW_DIR.glob(f'{sistema}_*.csv')):
        match = FILE_RE.match(path.name)
        if not match:
            continue
        _, sede, anio = match.groups()
        df = pd.read_csv(path)
        df['archivo'] = path.name
        df['sede'] = sede
        df['anio'] = int(anio)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

data_transacciones = cargar_sistema('transacciones')
data_clinico = cargar_sistema('clinica')
data_prescripciones = cargar_sistema('prescripciones')

pd.DataFrame({
    'sistema': ['transacciones', 'clinico', 'prescripciones'],
    'registros': [len(data_transacciones), len(data_clinico), len(data_prescripciones)],
    'columnas': [data_transacciones.shape[1], data_clinico.shape[1], data_prescripciones.shape[1]],
    'archivos': [data_transacciones['archivo'].nunique(), data_clinico['archivo'].nunique(), data_prescripciones['archivo'].nunique()]
})

## 3. Estandarizacion de nombres

En esta etapa se normalizan los nombres de pacientes para reducir diferencias de captura entre sistemas. La regla aplicada convierte el texto a minusculas, elimina acentos y remueve signos de puntuacion como puntos, comas y otros caracteres no alfabeticos. Tambien compacta espacios multiples. Esta transformacion no elimina la columna original; crea una columna nueva para conservar trazabilidad.

In [ ]:
def quitar_acentos(texto):
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(caracter for caracter in texto if not unicodedata.combining(caracter))

def estandarizar_nombre(nombre):
    if pd.isna(nombre):
        return pd.NA
    nombre = str(nombre).strip().lower()
    nombre = quitar_acentos(nombre)
    nombre = re.sub(r'[^a-zñ\s]', ' ', nombre)
    nombre = re.sub(r'\s+', ' ', nombre).strip()
    return nombre

ejemplos = pd.Series([
    'María Fernanda Hernández López',
    'M. Fernanda Hernández, López',
    'JUAN PÉREZ.',
    'Ana-García Ruiz'
])

pd.DataFrame({
    'nombre_original': ejemplos,
    'nombre_estandarizado': ejemplos.apply(estandarizar_nombre)
})

### 3.1 Aplicacion en transacciones

En transacciones, el campo de nombre del paciente es `nombre_paciente`. Se crea la columna `nombre_paciente_std` para conservar el valor original y tener una version estandarizada para comparaciones futuras.

In [ ]:
data_transacciones['nombre_paciente_std'] = data_transacciones['nombre_paciente'].apply(estandarizar_nombre)

data_transacciones[['nombre_paciente', 'nombre_paciente_std', 'archivo', 'sede', 'anio']].head(15)

### 3.2 Aplicacion en sistema clinico

En el sistema clinico, el campo de nombre del paciente es `nombre_completo`. Se crea `nombre_completo_std` para usarlo posteriormente en deteccion de duplicados y vinculacion con otros sistemas.

In [ ]:
data_clinico['nombre_completo_std'] = data_clinico['nombre_completo'].apply(estandarizar_nombre)

data_clinico[['nombre_completo', 'nombre_completo_std', 'archivo', 'sede', 'anio']].head(15)

### 3.3 Aplicacion en prescripciones

En prescripciones, el campo de nombre del paciente tambien es `nombre_completo`. Se crea `nombre_completo_std` para facilitar comparaciones contra el sistema clinico y transacciones.

In [ ]:
data_prescripciones['nombre_completo_std'] = data_prescripciones['nombre_completo'].apply(estandarizar_nombre)

data_prescripciones[['nombre_completo', 'nombre_completo_std', 'archivo', 'sede', 'anio']].head(15)

### 3.4 Validacion de la estandarizacion

Se revisa que los nombres estandarizados no contengan mayusculas, acentos ni signos de puntuacion. Los conteos siguientes deben ser cero si la normalizacion se aplico correctamente.

In [ ]:
patron_no_estandar = re.compile(r'[^a-zñ\s]')

def contar_nombres_no_estandarizados(serie):
    serie = serie.dropna().astype(str)
    return int(serie.str.contains(patron_no_estandar).sum())

validacion_nombres = pd.DataFrame({
    'sistema': ['transacciones', 'clinico', 'prescripciones'],
    'registros': [len(data_transacciones), len(data_clinico), len(data_prescripciones)],
    'nombres_no_estandarizados': [
        contar_nombres_no_estandarizados(data_transacciones['nombre_paciente_std']),
        contar_nombres_no_estandarizados(data_clinico['nombre_completo_std']),
        contar_nombres_no_estandarizados(data_prescripciones['nombre_completo_std'])
    ],
    'nombres_nulos': [
        data_transacciones['nombre_paciente_std'].isnull().sum(),
        data_clinico['nombre_completo_std'].isnull().sum(),
        data_prescripciones['nombre_completo_std'].isnull().sum()
    ]
})

validacion_nombres